In [ ]:
import os
import numpy as np
import pandas as pd

from gudhi.representations import PersistenceImage
from gudhi.representations.preprocessing import BirthPersistenceTransform
from scipy.ndimage import gaussian_filter

PI_RESOLUTION = (50, 50)
BANDWIDTH = 2
WEIGHT_MODE = "const"
IMG_SIGMA = 0
NORMALIZATION = "l1"
THR_MODE = "p10"

BASE = "RIPS"
DIM_INTERVALS = [0, 1]

EPS = 1e-12
BP = BirthPersistenceTransform()


def read_and_save(filedir, tube):
    if tube and tube[0] != '.':
        file_name, file_extension = os.path.splitext(os.path.join(filedir, tube))
        tubenamerips = tube.split('_')[-1].split('.')[0]

        if file_extension != '.pdf' and tubenamerips == 'Rips0':
            r0_path = os.path.join(filedir, '_'.join(tube.split('_')[:-1]) + '_Rips0.txt')
            r1_path = os.path.join(filedir, '_'.join(tube.split('_')[:-1]) + '_Rips1.txt')

            Rips0 = np.array(pd.read_csv(r0_path, sep=" ", header=None))
            if len(Rips0) and np.isinf(Rips0[-1, 1]):
                Rips0 = Rips0[:-1]

            Rips1 = np.array(pd.read_csv(r1_path, sep=" ", header=None))
            if len(Rips1) and np.isnan(Rips1[-1, 1]):
                Rips1[-1, 1] = 0

            return [[Rips0, Rips1], None, None]
    return []


def list_patients(root_dir):
    return [x for x in sorted(os.listdir(root_dir)) if not x.startswith(".")]


def find_rips0_file(patient_dir):
    for f in sorted(os.listdir(patient_dir)):
        if f.startswith("."):
            continue
        if f.endswith("_Rips0.txt"):
            return f
    return None


def load_group_diagrams(group_root):
    out = {}
    for patient in list_patients(group_root):
        p_dir = os.path.join(group_root, patient)
        rips0_file = find_rips0_file(p_dir)
        if rips0_file is None:
            continue
        data = read_and_save(p_dir, rips0_file)
        if not data:
            continue
        diagrams = data[0]
        if diagrams is None or len(diagrams) < 2:
            continue
        out[patient] = diagrams
    return out


def apply_persistence_threshold(pairs, thr):
    if pairs is None or len(pairs) == 0:
        return np.empty((0, 2), dtype=float)

    pairs = np.asarray(pairs, dtype=float)
    if pairs.ndim != 2 or pairs.shape[1] < 2:
        return np.empty((0, 2), dtype=float)

    b = pairs[:, 0]
    d = pairs[:, 1]
    pers = d - b

    keep = (
        np.isfinite(b) &
        np.isfinite(d) &
        (d > b) &
        (pers >= thr)
    )
    return pairs[keep]


def compute_global_thresholds(diag_nr, diag_r, dim_intervals, thr_mode="0"):
    thr_dim = {}
    if thr_mode == "0":
        for d in dim_intervals:
            thr_dim[d] = 0.0
        return thr_dim

    all_diags = list(diag_nr.values()) + list(diag_r.values())
    for d in dim_intervals:
        pers_all = []
        for diagrams in all_diags:
            pairs = diagrams[d]
            if pairs is None or len(pairs) == 0:
                continue
            pairs = np.asarray(pairs, float)
            if pairs.ndim != 2 or pairs.shape[1] < 2:
                continue
            b = pairs[:, 0]
            dth = pairs[:, 1]
            p = dth - b
            mask = np.isfinite(b) & np.isfinite(dth) & (dth > b) & (p > 0)
            p = p[mask]
            if p.size:
                pers_all.append(p)
        if pers_all:
            pers_cat = np.concatenate(pers_all)
            thr_dim[d] = float(np.percentile(pers_cat, 10))
        else:
            thr_dim[d] = 0.0
    return thr_dim


def make_weight(mode):
    if mode == "const":
        return lambda x: 1.0
    if mode == "pers":
        return lambda x: max(float(x[1]), 0.0)
    raise ValueError("Unknown weight mode: " + str(mode))


def normalize_vec(v, mode):
    if mode == "none":
        return v
    if mode == "l1":
        s = np.sum(np.abs(v))
        return v / (s + EPS)
    raise ValueError("Unknown normalization: " + str(mode))


def smooth_image_vector(v, resolution, sigma):
    if sigma <= 0:
        return v
    nx, ny = resolution
    img = v.reshape(nx, ny)
    img = gaussian_filter(img, sigma=float(sigma), mode="nearest")
    return img.ravel()

NR_root = os.path.join(BASE, "NonRelapse")
R_root  = os.path.join(BASE, "Relapse")

NR_diagrams = load_group_diagrams(NR_root)
R_diagrams  = load_group_diagrams(R_root)

thr_dim = compute_global_thresholds(NR_diagrams, R_diagrams, DIM_INTERVALS, THR_MODE)


pi_dim = {}
nx, ny = PI_RESOLUTION

for d in DIM_INTERVALS:
    diags_bp = []
    for diagrams in list(NR_diagrams.values()) + list(R_diagrams.values()):
        pairs = diagrams[d]
        f_pairs = apply_persistence_threshold(pairs, thr_dim[d])
        if f_pairs is None or len(f_pairs) == 0:
            continue
        diags_bp.append(BP(np.asarray(f_pairs, float)))

    pi = PersistenceImage(
        bandwidth=float(BANDWIDTH),
        weight=make_weight(WEIGHT_MODE),
        resolution=list(PI_RESOLUTION),
        im_range=[np.nan, np.nan, np.nan, np.nan],
    )
    if len(diags_bp) > 0:
        pi.fit(diags_bp)

    pi_dim[d] = pi

# NON RELAPSE
PIH0H1_NR = []
listdirNR = sorted(NR_diagrams.keys())

for patient in listdirNR:
    diagrams = NR_diagrams[patient]

    feat_list = []
    for d in DIM_INTERVALS:
        thr = thr_dim[d]
        pi = pi_dim[d]
        pairs = diagrams[d]
        f_pairs = apply_persistence_threshold(pairs, thr)

        if f_pairs is None or len(f_pairs) == 0:
            v = np.zeros(nx * ny, float)
        else:
            diag_bp = BP(np.asarray(f_pairs, float))
            v = pi(diag_bp).astype(float)

        v = normalize_vec(v, NORMALIZATION)
        v = smooth_image_vector(v, PI_RESOLUTION, IMG_SIGMA)
        feat_list.append(v)

    feat = np.concatenate(feat_list, axis=0)
    PIH0H1_NR.append(feat)

# RELAPSE
PIH0H1_R = []
listdirR = sorted(R_diagrams.keys())

for patient in listdirR:
    diagrams = R_diagrams[patient]

    feat_list = []
    for d in DIM_INTERVALS:
        thr = thr_dim[d]
        pi = pi_dim[d]
        pairs = diagrams[d]
        f_pairs = apply_persistence_threshold(pairs, thr)

        if f_pairs is None or len(f_pairs) == 0:
            v = np.zeros(nx * ny, float)
        else:
            diag_bp = BP(np.asarray(f_pairs, float))
            v = pi(diag_bp).astype(float)

        v = normalize_vec(v, NORMALIZATION)
        v = smooth_image_vector(v, PI_RESOLUTION, IMG_SIGMA)
        feat_list.append(v)

    feat = np.concatenate(feat_list, axis=0)
    PIH0H1_R.append(feat)

folder = "PersistenceImages01"
subfolder = os.path.join(BASE, folder)

os.makedirs(os.path.join(subfolder, "Relapse"), exist_ok=True)
os.makedirs(os.path.join(subfolder, "NonRelapse"), exist_ok=True)

for i, curve in enumerate(PIH0H1_R):
    np.savetxt(os.path.join(subfolder, "Relapse", f"{listdirR[i]}.csv"), curve)

for i, curve in enumerate(PIH0H1_NR):
    np.savetxt(os.path.join(subfolder, "NonRelapse", f"{listdirNR[i]}.csv"), curve)